# Practica 3 de introducción a la inteligencia artificial

## Importar librerias necesarias y obtener datos

In [29]:
import kagglehub
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn


# Download latest version
path = kagglehub.dataset_download("dineshpiyasamara/geometric-shapes-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\manue\.cache\kagglehub\datasets\dineshpiyasamara\geometric-shapes-dataset\versions\1


## Comprobar si nuestro dataset cumple con tener mas de 1000 imagenes

## Preprosesamiento de imagenes

In [30]:


IMAGE_SIZE = 64
BATCH_SIZE = 128

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3) 
])

dataset = datasets.ImageFolder(path, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Total imágenes cargadas:", len(dataset))


Total imágenes cargadas: 30000


## Arquitectura DCGAN

### Generador

In [31]:

Z_DIM = 100  # Espacio latente

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(Z_DIM, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)


### Discriminador

In [32]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


## Configuración del entrenamiento

In [33]:
device = "cuda" if torch.cuda.is_available() else "cpu"

gen = Generator().to(device)
disc = Discriminator().to(device)

criterion = nn.BCELoss()
lr = 0.0002

opt_gen = torch.optim.Adam(gen.parameters(), lr=lr, betas=(0.5, 0.999))
opt_disc = torch.optim.Adam(disc.parameters(), lr=lr, betas=(0.5, 0.999))

EPOCHS = 50


### Loops de entrenamiento

In [ ]:
import os
from torchvision.utils import save_image

save_dir = "gan_training"
os.makedirs(save_dir, exist_ok=True)

EPOCHS = 30

for epoch in range(EPOCHS):
    for i, (real, _) in enumerate(dataloader):
        real = real.to(device)
        batch_size = real.size(0)

        # -------- Entrenar Discriminador --------
        noise = torch.randn(batch_size, Z_DIM, 1, 1, device=device)
        fake = gen(noise)

        disc_real = disc(real).view(-1)
        loss_real = criterion(disc_real, torch.ones_like(disc_real))

        disc_fake = disc(fake.detach()).view(-1)
        loss_fake = criterion(disc_fake, torch.zeros_like(disc_fake))

        loss_disc = (loss_real + loss_fake) / 2

        opt_disc.zero_grad()
        loss_disc.backward()
        opt_disc.step()

        # -------- Entrenar Generador --------
        output = disc(fake).view(-1)
        loss_gen = criterion(output, torch.ones_like(output))

        opt_gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()

    # ---- Guardar imagen del epoch ----
    noise = torch.randn(25, Z_DIM, 1, 1).to(device)
    fake_samples = gen(noise)
    fake_samples = (fake_samples + 1) / 2  # desnormalizar
    save_image(fake_samples, f"{save_dir}/epoch_{epoch+1}.png", nrow=5)

    print(f"[Epoch {epoch+1}/{EPOCHS}] Loss_D: {loss_disc:.4f} | Loss_G: {loss_gen:.4f}")

    # ---- Guardar checkpoint cada 10 epochs ----
    if (epoch + 1) % 10 == 0:
        torch.save(gen.state_dict(), f"{save_dir}/gen_epoch_{epoch+1}.pth")
        torch.save(disc.state_dict(), f"{save_dir}/disc_epoch_{epoch+1}.pth")
